# 02 · Gaussian Actor、联合密度与梯度

## 学习目标

用两维动作看清 `.sum(-1)`；确认单样本 Critic 输出仍是 `[1]`。

本 Notebook 直接运行已补全的 exercise 实现；修改 `.py` 后重新运行相关 Cell。

## 环境准备

先按 [使用指南](README.md) 安装 `.[dev,notebook]`，选择项目 Kernel。

In [1]:
from pathlib import Path
import os
import sys
import torch
import numpy as np

# 从仓库根目录或 notebooks/ 打开都可；不要依赖某台机器的绝对路径。
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'pyproject.toml').exists()
                     and (p / 'src/ppo_pendulum').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('请从项目目录启动 JupyterLab')
os.chdir(PROJECT_ROOT)
%load_ext autoreload
%autoreload 2
torch.manual_seed(7)
np.random.seed(7)
print('Python:', sys.executable)
print('Project:', PROJECT_ROOT.name)

Python: /Users/matt/Documents/Codex/2026-11-15/referenced-chatgpt-conversation-this-is-an-2/work/.venv/bin/python
Project: ppo-inverted-pendulum


## 1. 构造两维动作例子

这里暂时用 `action_dim=2`，更容易观察联合 log density。实际倒立摆是 1 维动作。

In [2]:
from ppo_pendulum.exercises.actor_critic import ExerciseActorCritic
model = ExerciseActorCritic(obs_dim=4, action_dim=2)
observations = torch.zeros(3, 4)
dist = model.distribution(observations)
actions = dist.sample()
per_dim_log_prob = dist.log_prob(actions)
joint_log_prob = model.actor.joint_log_prob(dist, actions)
print('actions:', actions)
print('per dimension:', per_dim_log_prob)
print('joint:', joint_log_prob)
assert per_dim_log_prob.shape == (3, 2)
assert joint_log_prob.shape == (3,)
torch.testing.assert_close(joint_log_prob, per_dim_log_prob.sum(-1))
assert model.value(torch.zeros(1, 4)).shape == (1,)

actions: tensor([[ 0.7391, -1.1856],
        [-0.5936,  0.3279],
        [-0.2129,  0.5983]])
per dimension: tensor([[-1.4190, -2.1264],
        [-0.7255, -0.6285],
        [-0.4310, -1.0166]], grad_fn=<SubBackward0>)
joint: tensor([-3.5453, -1.3540, -1.4476], grad_fn=<SumBackward1>)


## 2. 同一组动作与固定优势的策略梯度

`actions` 和 `advantages` 是已采集数据；梯度流经当前策略的 log probability。共享一个优势并不意味着各动作维的参数梯度相同。

In [3]:
advantages = torch.tensor([1.0, -0.5, 0.25])
loss = -(joint_log_prob * advantages).mean()
model.zero_grad()
loss.backward()
print('Actor loss:', loss.item())
print('log_std gradient:', model.actor.log_std.grad)
assert torch.isfinite(model.actor.log_std.grad).all()
assert all(p.grad is None for p in model.critic_parameters())

Actor loss: 1.0767472982406616
log_std gradient: tensor([-0.3165, -0.9181])


## 3. 验证拆分实现

检查 exercise Actor/Critic 的联合概率和单样本输出形状。

In [4]:
import subprocess
subprocess.run([sys.executable, '-m', 'pytest', '-q',
                'checks/test_exercises.py', '-k', 'stage_2'], check=True)
print('Actor/Critic exercise checks passed.')

.                                                                        [100%]


Actor/Critic exercise checks passed.


## 检查

改变 batch 大小和动作维度。联合 log probability 应始终是 `[B]`；不要把 batch 维求和，也不要用无参数 `.squeeze()`。

## 下一步

打开 `03_gae_and_bootstrap.ipynb`，学习 Actor 的优势和 Critic 的 target 从哪里来。

公式对照：[PPO_FORMULAS.md](../docs/PPO_FORMULAS.md)。